In [1]:
import pandas as pd
loans_raw = pd.read_csv('../data/loans.csv')

# 1)

In [2]:
loans = loans_raw.copy()
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


In [3]:
loans.shape

(623, 9)

problems noted already:
1) notes has 0 non-null values - useless column
2) loan_date is a str - not a datetime
3) return_date is a str - not a datetime
4) late_fee is a str - not a float
5) returned is a str - not a bool

# 2)

In [4]:
loans.genre.nunique()

28

In [5]:
loans.genre.head(8).tolist()

[' ROMANCE',
 'Crime',
 'fantasy',
 'fantasy',
 'CRIME',
 'Children ',
 'fantasy',
 'children']

In [6]:
genre_cleaned = loans.genre.str.strip().str.lower().str.title()
genre_cleaned.value_counts()

genre
Crime              163
Fantasy             90
Romance             86
Children            76
Children'S          59
History             59
Science Fiction     33
Sci-Fi              30
Sci Fi              27
Name: count, dtype: int64

In [7]:
genre_cleaned = (
    loans.genre
    .str.strip()
    .str.lower()
    .str.title()
    .replace("Children'S", "Children")
    .replace("Sci Fi", "Science Fiction")
    .replace("Sci-Fi", "Science Fiction")
)
genre_cleaned.value_counts()

genre
Crime              163
Children           135
Fantasy             90
Science Fiction     90
Romance             86
History             59
Name: count, dtype: int64

In [8]:
loans.genre = genre_cleaned
loans.genre.nunique()

6

# 3)

In [9]:
loans.returned.nunique()

10

In [10]:
print(loans.returned.isna().sum())

0


In [11]:
loans.returned.value_counts()

returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64

In [12]:
loans.returned = loans.returned.str.lower().map({'yes': True, 'y': True, 'no': False, 'n': False})

In [13]:
loans.returned.value_counts()

returned
True     593
False     30
Name: count, dtype: int64

In [14]:
print(loans.returned.isna().sum())

0


# 4)

In [15]:
loans.late_fee.astype(float)

ValueError: could not convert string to float: '7,50 kr'

In [16]:
late_fee_cleaned = (
    loans.late_fee
    .str.replace('kr', '')
    .str.replace(' ', '')
    .str.replace(',', '.')
)

In [17]:
loans.late_fee = late_fee_cleaned.astype(float)

In [18]:
loans.late_fee.describe()

count    623.000000
mean      12.323435
std       16.106338
min        0.000000
25%        0.000000
50%        0.000000
75%       25.000000
max       47.500000
Name: late_fee, dtype: float64

In [19]:
print(loans.late_fee.sum())

7677.5


# 5)

In [20]:
dates = pd.to_datetime(loans.loan_date, errors='coerce')
dates.isna().sum()

np.int64(3)

In [21]:
loans.loc[dates.isna(), ['loan_id', 'loan_date']]

,loan_id,loan_date
477,L-30463,unknown
483,L-30468,2026-02-30
592,L-30572,unknown


In [22]:
loans.loan_date = dates

In [23]:
loans.return_date = pd.to_datetime(loans.return_date, errors='coerce')

In [24]:
print(loans.loan_date.min())
print(loans.loan_date.max())

2026-01-05 00:00:00
2026-06-27 00:00:00


In [25]:
loans.loan_date.dt.to_period('M').value_counts().sort_index()

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64

In [26]:
print(loans.loan_date.dt.to_period('M').value_counts().sum())
print(loans.loan_id.count())

620
623


# 6)

In [27]:
print(loans_raw.duplicated().sum())

12


In [28]:
print(loans.duplicated().sum())

18


data were different previously may be duplicate now since we stripped spaces, normalized data etc

In [29]:
print(loans_raw['loan_id'].duplicated().sum())

18


In [30]:
print(loans['loan_id'].duplicated().sum())

18


In [31]:
raw_late_fees_sum = loans.late_fee.sum()

In [32]:
loans = loans.drop_duplicates().reset_index(drop=True)

In [33]:
print(raw_late_fees_sum - loans.late_fee.sum())

230.0


In [ ]:
print(round((raw_late_fees_sum - loans.late_fee.sum())/(loans.late_fee.sum()), 4))

0.0309


# 7)

In [34]:
loans.drop(columns='notes')

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
0,L-30001,M0150,A Second Spring,Romance,2026-01-06,2026-01-30,7.5,True
1,L-30002,M0037,The Silent Harbour,Crime,2026-01-05,2026-01-25,0.0,True
2,L-30003,M0143,The Glass Mage,Fantasy,2026-01-05,2026-01-26,0.0,True
3,L-30004,M0121,Dead Reckoning Lane,Crime,2026-01-05,2026-01-12,0.0,True
4,L-30005,M0109,Ollie Learns to Swim,Children,2026-01-06,2026-02-02,15.0,True
...,...,...,...,...,...,...,...,...
600,L-30596,M0104,Mimmi and the Moon,Children,2026-06-27,2026-06-25,0.0,True
601,L-30597,M0012,Kings of the North Sea,History,2026-06-26,2026-07-27,25.0,True
602,L-30598,M0071,Shadow of the Tower,Fantasy,2026-06-27,2026-08-03,40.0,True
603,L-30599,M0113,The Dragon Who Was Afraid,Children,2026-06-27,2026-07-16,0.0,True


In [35]:
loans2 = loans_raw.copy()
loans2.genre = (
    loans2.genre
    .str.strip()
    .str.lower()
    .str.title()
    .replace("Children'S", "Children")
    .replace("Sci Fi", "Science Fiction")
    .replace("Sci-Fi", "Science Fiction")
)
loans2.returned = loans2.returned.str.lower().map({'yes': True, 'y': True, 'no': False, 'n': False})
loans2.late_fee = (
    loans2.late_fee
    .str.replace('kr', '')
    .str.replace(' ', '')
    .str.replace(',', '.')
).astype(float)
loans2.loan_date = pd.to_datetime(loans2.loan_date, errors='coerce')
loans2.return_date = pd.to_datetime(loans2.return_date, errors='coerce')
loans2 = loans2.drop_duplicates().reset_index(drop=True)
loans2.shape

(605, 9)

In [ ]:
loans2.to_csv('../data/loans_clean.csv', index=False)